# DFedSET 超参数分析

In [ ]:
import os
import matplotlib.ticker as mticker
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown, HTML
from utils import ResultLoader
from matplotlib.colors import ListedColormap, BoundaryNorm

# 设置绘图风格
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 16,
    'axes.labelsize': 14,
    'figure.titlesize': 18
})

# 初始化加载器，指向 Ray 实验结果目录
loader = ResultLoader("../results_ray")

In [ ]:
# 实验配置
common_args = {
    "dataset": "cifar10",
    "partition": "dirichlet",
    "num_clients": 20,
    "alpha": 0.1,
    "epochs": 1,
    "batch_size": 64,
    "lr": 0.01,
    "adj_type": "ring",
}
algorithms_comm = {
    "Local":     {"algo": "local",   "kwargs": {}},
    "FedAvg":    {"algo": "fedavg",   "kwargs": {}},
    "FedProto":  {"algo": "fedproto", "kwargs": {"mu": 0.1}},
    "ProxyFL":   {"algo": "proxyfl", "kwargs": {"mu": 0.01}},

    "DFedAvgM":  {"algo": "dfedavgm", "kwargs": {}},
    "PearFL":    {"algo": "pearfl",  "kwargs": {"lamda": 0.1}},
    "EF-HC":     {"algo": "efhc",   "kwargs": {"event_r": 250, "bandwidth_mean": 5000, "bandwidth_std": 0.0}},
    "L2C":       {"algo": "l2c",     "kwargs": {"val_ratio": 0.1, "lr_alpha": 0.1, "prune_round": 20, "prune_num": 0}},
    "DFedPGP":   {"algo": "dfedpgp", "kwargs": {"local_v_epochs": 1, "lr_v": 0.01, "momentum_v": 0.0, "weight_decay_v": 0.0}},
    "DisPFL":    {"algo": "dispfl",  "kwargs": {"dense_ratio": 0.5, "anneal_factor": 0.5, "erk_power_scale": 1.0}},

    "DFedSET":   {"algo": "dfedset", "kwargs": {"lambda_sa": 20.0, "eta": 0.9, "lambda_so": 5}},
    "DFedSET (All-Trigger)": {"algo": "dfedset", "kwargs": {"lambda_sa": 20.0, "eta": 0.9, "lambda_so": 5, "ablate_name": "trigger_all"}},
}
plot_fontsize = 22
plot_linewidth = 2
marker_size = 10
args_dfedset = {**common_args, **algorithms_comm["DFedSET"]["kwargs"]}

def calc_cum_comm(data, algo, num_clients, join_ratio):
    """返回累计通信量（累计参数传输数量）"""
    acc_dict = data.get("acc", {})
    if isinstance(acc_dict, dict) and "model" in acc_dict:
        n_rounds = len(acc_dict["model"])
    else:
        n_rounds = len(acc_dict) if isinstance(acc_dict, list) else 0

    if algo == "dfedset":
        num_tr = data.get("num_triggered", [])
        per_round = np.array(num_tr[:n_rounds]) * BODY_PARAMS + num_clients * DFEDSET_EXTRA
    elif algo == "efhc":
        num_tr = data.get("num_triggered", [])
        per_round = np.array(num_tr[:n_rounds]) * MODEL_PARAMS
    elif algo == "local":
        per_round = np.zeros(n_rounds)
    elif algo == "l2c":
        per_round = np.full(n_rounds, num_clients * 2 * MODEL_PARAMS)
    elif algo == "dispfl":
        per_round = np.full(n_rounds, int(num_clients * MODEL_PARAMS * 0.5))
    elif algo == "dfedpgp":
        per_round = np.full(n_rounds, int(num_clients * BODY_PARAMS))
    elif algo == "fedproto":
        per_round = np.full(n_rounds, int(num_clients * PROTOS_PARAMS))
    elif algo == "pearfl":
        per_round = np.full(n_rounds, int(num_clients * (MODEL_PARAMS + PROTOS_PARAMS)))
    else:
        per_round = np.full(n_rounds, int(num_clients * join_ratio * MODEL_PARAMS))
    return np.cumsum(per_round)

In [ ]:
# ====== 可配置参数 ======
ALGO = "dfedset"          # 支持："local", "fedavg", "fedproto", "proxyfl", "dfedavgm", "pearfl", "efhc", "l2c", "dfedpgp", "dispfl", "dfedset"
DATASET = "cifar10"       # 支持："cifar10", "cifar100", "cinic10"
PARTITION = "dirichlet"   # 支持："dirichlet", "pathological"
SPECIFIC_RUN = 0          # 运行序列：0, 1, 2, 3, 4

# 算法特定参数覆盖（如果为 None，则默认使用 algorithms_comm 中的配置）
# 例如对于 dfedset: {"lambda_sa": 20.0, "eta": 0.9, "lambda_so": 5.0}
# 对于 pearfl: {"lamda": 0.1}
CUSTOM_PARAMS = {"lambda_sa": 20.0, "eta": 0.9, "lambda_so": 5.0}

# 消融实验名称（仅 dfedset 有效，例如 "relay", "aggregator", "confidence_count" 等，普通实验设为 None）
ABLATE_NAME = None
# ========================

# 获取默认的基础参数
run_args = common_args.copy()
run_args["num_clients"] = 20
run_args["dataset"] = DATASET
run_args["partition"] = PARTITION
if PARTITION == "dirichlet":
    run_args["alpha"] = 0.1
    if "n_class" in run_args: del run_args["n_class"]
else:
    run_args["n_class"] = 0
    if "alpha" in run_args: del run_args["alpha"]

# 匹配对应算法的超参数
algo_matched = None
for k, v in algorithms_comm.items():
    if v["algo"].lower() == ALGO.lower():
        algo_matched = v
        break

if algo_matched is not None:
    algo_kwargs = algo_matched["kwargs"].copy()
else:
    algo_kwargs = {}

if CUSTOM_PARAMS is not None:
    algo_kwargs.update(CUSTOM_PARAMS)

args = {**run_args, **algo_kwargs}
data = loader.load(ALGO.lower(), ablate_name=ABLATE_NAME, **args, specific_run=SPECIFIC_RUN, keys=["acc"])
if data is None:
    raise RuntimeError(f"未找到匹配实验结果: algo={ALGO}, dataset={DATASET}, partition={PARTITION}, params={algo_kwargs}, run={SPECIFIC_RUN}")

acc_dict = data.get("acc", {})
if isinstance(acc_dict, dict):
    acc = acc_dict.get("model", acc_dict.get("local", []))
else:
    acc = acc_dict

# 构造标题
param_str = ", ".join([f"{k}={v}" for k, v in algo_kwargs.items()])
title = f"{ALGO.upper()} on {DATASET.upper()}({PARTITION})\n{param_str} (Run {SPECIFIC_RUN})"
if ABLATE_NAME:
    title += f" (Ablation: {ABLATE_NAME})"

plt.figure(figsize=(10, 7))
plt.plot(acc, linewidth=plot_linewidth, color="#D62728")
plt.xlabel("Round", fontsize=plot_fontsize)
plt.ylabel("Accuracy (%)", fontsize=plot_fontsize)
plt.tick_params(labelsize=plot_fontsize)
plt.title(title, fontsize=plot_fontsize - 4)
plt.grid(alpha=0.3)
plt.tight_layout()
os.makedirs("figures", exist_ok=True)
plt.savefig(f"figures/{ALGO}_{DATASET}_{PARTITION}_acc_curve.pdf", bbox_inches="tight")
plt.show()

## Table 1: 多算法精度对比

**跨 CIFAR-10 / CIFAR-100 / CINIC-10 × Dirichlet (α=0.1 / 0.5) / Pathological，5 次实验的 Max Acc Mean ± Std。**
**加粗** 为最优，**下划线** 为第二。

In [ ]:
# Table 1: 多算法精度对比
table1_datasets = ["cifar10", "cinic10", "cifar100"]
table1_cols = [
    ("dirichlet", 0.1, "Dir. 0.1"),
    ("dirichlet", 0.5, "Dir. 0.5"),
    ("pathological", None, "Path."),
]
ds_disp = {"cifar10": "CIFAR-10", "cifar100": "CIFAR-100", "cinic10": "CINIC-10"}
table1_algo_order = [
    "Local", "FedAvg", "DFedAvgM", "PearFL",
    "FedProto", "ProxyFL", "EF-HC", "L2C", "DFedPGP", "DisPFL", "DFedSET",
]

def get_acc_max(data):
    acc = data.get("acc", {})
    if isinstance(acc, dict):
        for key in ["model", "local"]:
            if key in acc:
                return max(acc[key])
    elif isinstance(acc, list):
        return max(acc)
    return None

results = {}
for algo_label in table1_algo_order:
    algo_spec = algorithms_comm[algo_label]
    algo_name, algo_kwargs = algo_spec["algo"], algo_spec["kwargs"]
    results[algo_label] = {}
    specs = []
    for ds in table1_datasets:
        for part, alpha, _ in table1_cols:
            part_kw = {"alpha": alpha} if part == "dirichlet" else {"n_class": 0}
            load_args = dict(dataset=ds, partition=part, num_clients=20, epochs=1,
                batch_size=64, lr=0.01, adj_type="ring", **part_kw, **algo_kwargs)
            for s in range(5):
                specs.append({**load_args, "specific_run": s})
    for ds in table1_datasets:
        for part, alpha, _ in table1_cols:
            part_kw = {"alpha": alpha} if part == "dirichlet" else {"n_class": 0}
            load_args = dict(dataset=ds, partition=part, num_clients=20, epochs=1,
                batch_size=64, lr=0.01, adj_type="ring", **part_kw, **algo_kwargs)
            key = (ds, part, alpha)
            accs = []
            for s in range(5):
                data = loader.load(algo_name, **load_args, specific_run=s, keys=["acc"])
                if data is None:
                    continue
                v = get_acc_max(data)
                if v is not None:
                    accs.append(v)
            if accs:
                results[algo_label][key] = (np.mean(accs), np.std(accs), len(accs))
            else:
                results[algo_label][key] = None

col_best, col_second = {}, {}
for ds in table1_datasets:
    for part, alpha, _ in table1_cols:
        key = (ds, part, alpha)
        entries = [(a, results[a][key]) for a in table1_algo_order if results[a].get(key) is not None]
        if entries:
            sorted_entries = sorted(entries, key=lambda x: x[1][0], reverse=True)
            col_best[key] = sorted_entries[0][0]
            col_second[key] = sorted_entries[1][0] if len(sorted_entries) > 1 else None
        else:
            col_best[key] = col_second[key] = None

def _fmt_cell(data, best, second, label):
    if data is None:
        return "—", False, False
    mean, s, n = data
    text = "%.2f ± %.2f" % (mean, s)
    return text, (best == label), (second == label)

# --- HTML display ---
html = "<table style=\"margin:0 auto;border-collapse:collapse;\" border=\"1\" class=\"dataframe\">\n"
html += "<thead>\n<tr>\n"
html += "<th rowspan=\"2\" style=\"text-align:center;\">Algorithm</th>\n"
for ds in table1_datasets:
    html += f"<th colspan=\"3\" style=\"text-align:center;\">{ds_disp[ds]}</th>\n"
html += "</tr>\n<tr>\n"
for ds in table1_datasets:
    for _, _, disp in table1_cols:
        html += f"<th style=\"text-align:center;\">{disp}</th>\n"
html += "</tr>\n</thead>\n<tbody>\n"
for algo_label in table1_algo_order:
    html += "<tr>\n"
    html += f"<td style=\"text-align:left;font-weight:bold;\">{algo_label}</td>\n"
    for ds in table1_datasets:
        for part, alpha, _ in table1_cols:
            key = (ds, part, alpha)
            text, is_best, is_sec = _fmt_cell(results[algo_label].get(key), col_best.get(key), col_second.get(key), algo_label)
            style = "text-align:center;"
            if is_best:
                style += "font-weight:bold;"
            elif is_sec:
                style += "text-decoration:underline;"
            html += f"<td style=\"{style}\">{text}</td>\n"
    html += "</tr>\n"
html += "</tbody>\n</table>"
display(Markdown("### Table 1: 多算法精度对比"))
display(HTML(html))

# --- LaTeX output ---
print("\nLaTeX format:")
col_spec = "l" + "ccc" * len(table1_datasets)
print("\\begin{table}[htpb]")
print("\\centering\\small")
print("\\begin{tabular}{" + col_spec + "}")
print("\\toprule")
h1 = "Algorithm"
for ds in table1_datasets:
    h1 += " & \\multicolumn{3}{c}{" + ds_disp[ds] + "}"
print(h1 + " \\\\")
h2 = ""
for ds in table1_datasets:
    for _, _, disp in table1_cols:
        h2 += " & " + disp
print(h2 + " \\\\")
print("\\midrule")
for algo_label in table1_algo_order:
    cells = [algo_label]
    for ds in table1_datasets:
        for part, alpha, _ in table1_cols:
            key = (ds, part, alpha)
            cell = results[algo_label].get(key)
            if cell is None:
                cells.append("---")
            else:
                mean, s, n = cell
                text = "$\\mathbf{%.2f \\pm %.2f}$" % (mean, s) if col_best.get(key) == algo_label else \
                       "$\\underline{%.2f \\pm %.2f}$" % (mean, s) if col_second.get(key) == algo_label else \
                       "%.2f $\\pm$ %.2f" % (mean, s)
                cells.append(text)
    print(" & ".join(cells) + " \\\\")
print("\\bottomrule")
print("\\end{tabular}")
print("\\end{table}")

## Table 2: 消融实验

**所需实验（共 10 组，每个维度独立消融，其他保持默认）：**

| 维度 | 变量 | 取值 |
|---|---|---|
| A-Relay | `ablate.relay` | `true`(默认), `false` |
| B-Aggregator | `ablate.aggregator` | `true`(默认·MH redirect), `false`(plain avg) |
| C-Confidence | `ablate.confidence` | `"log"`(默认), `"count"`, `"none"` |
| D-Trigger | `ablate.trigger` | `"adaptive"`(默认), `"global"`, `"all"` |

固定 λ_sa=20.0, λ_so=0.1, epochs=1

In [ ]:
# Ablation experiment config: ablate_name maps to subdirectory name
ablation_configs = {
    "Base":                    {"ablate_name": None, "category": "DFedSET"},
    "-":                       {"ablate_name": "relay", "category": "Relay"},
    "Plain Aggr.":             {"ablate_name": "aggregator", "category": "Aggregation"},
    "count":                   {"ablate_name": "confidence_count", "category": "Weighting"},
    "none":                    {"ablate_name": "confidence_none", "category": "Weighting"},
    "global (0.001)":          {"ablate_name": "trigger_global_gamma_0.001", "category": "Trigger"},
    "global (0.005)":          {"ablate_name": "trigger_global_gamma_0.005", "category": "Trigger"},
    "all":                     {"ablate_name": "trigger_all", "category": "Trigger"},
}

ablation_results = {}
for label, cfg in ablation_configs.items():
    seed_maxes = []
    seed_tr = []

    for s in range(5):
        data = loader.load("dfedset", ablate_name=cfg["ablate_name"],
                           **args_dfedset, specific_run=s, keys=["acc","num_triggered"])
        if data is None:
            continue
        # seed_maxes.append(max(data["acc"]["model"]))
        acc_data = data["acc"]
        if isinstance(acc_data, dict):
            acc_data = acc_data.get("model", acc_data.get("local", []))
        seed_maxes.append(max(acc_data))
        if "num_triggered" in data:
            tr = np.mean(data["num_triggered"]) / args_dfedset["num_clients"] * 100
        else:
            tr = 100.0
        seed_tr.append(tr)

    if not seed_maxes:
        continue

    m_max = np.mean(seed_maxes)
    trig_rate = np.mean(seed_tr)

    ablation_results[label] = {
        "category": cfg["category"],
        "model_max": m_max,
        "trig_rate": trig_rate
    }

# Build DataFrame
df_data = []
for label, res in ablation_results.items():
    df_data.append({
        "Component": res["category"],
        "Configuration": label,
        "Acc": f"{res['model_max']:.2f}",
        "Trigger Rate": f"{res['trig_rate']:.2f}"
    })
df = pd.DataFrame(df_data)
display(Markdown("### Ablation Study Results"))
display(df)

print("\nLaTeX format:")
print("\\begin{table}[htbp]")
print("\\centering")
print(df.to_latex(index=False, column_format="cccc"))
print("\\end{table}")

## Fig 2: 通信量 VS Acc，多算法对比

**所需实验：** 每种算法至少一个完整 run（1000 rounds），epochs 对齐。
所有算法同模型（cnn），同 join_ratio（1.0）。

- dfedset: epochs=1, λ_sa=20.0, λ_so=5
- efhc: epochs=1, event_r=250, bandwidth_mean=5000
- l2c, dispfl, pearfl, dfedavgm, dfedpgp, proxyfl, local, fedproto: epochs=1

In [ ]:
MODEL_PARAMS = 2122186     # 完整 CNN 参数量
BODY_PARAMS = 2117056      # DFedPGP 提取器（不含分类头）
PROTOS_PARAMS = 5120       # 原型 [10, 512]
DFEDSET_EXTRA = 5130       # S[10,512] + W[10]
PARAM_TO_GB = 4 / (1024 * 1024 * 1024)  # 每个参数 4 bytes → GB
algorithms_comm["DFedSET"] = {"algo": "dfedset", "kwargs": {"lambda_sa": 20.0, "eta": 0.9, "lambda_so": 5}}
# 选择需要绘制的算法
plot_algorithms = [
    "DFedSET",
    "DFedSET (All-Trigger)",
    "EF-HC",
    "DFedAvgM",
    "DisPFL",
    # "L2C",
    "PearFL",
    "DFedPGP",
    # "ProxyFL",
]
# 自定义各算法曲线的颜色和线型
line_styles = {
    "DFedSET":   {"color": "#D62728", "linestyle": "-"},
    "DFedSET (All-Trigger)": {"color": "#17BECF", "linestyle": "-"},
    "EF-HC":     {"color": "#1F77B4", "linestyle": "-"},
    "DFedAvgM":  {"color": "#9467BD", "linestyle": "-"},
    "DFedPGP":   {"color": "#FF7F0E", "linestyle": "-"},
    "PearFL":    {"color": "#2CA02C", "linestyle": "-"},
    "FedAvg":    {"color": "black", "linestyle": "-."},
    "Local":     {"color": "black", "linestyle": "--"},
    "ProxyFL":   {"color": "#8C564B", "linestyle": "-"},
    "L2C":       {"color": "#E377C2", "linestyle": "-"},
}

plt.figure(figsize=(10, 8))
for label, spec in algorithms_comm.items():
    if plot_algorithms and label not in plot_algorithms:
        continue
    args = {**common_args, **spec["kwargs"]}
    data = loader.load(spec["algo"], **args, specific_run=0, keys=["acc","num_triggered"])
    if data is None:
        print(f"  [跳过] {label}: 无结果")
        continue

    acc_dict = data.get("acc", {})
    if isinstance(acc_dict, dict) and "model" in acc_dict:
        acc = acc_dict["model"]
    elif isinstance(acc_dict, list):
        acc = acc_dict
    else:
        continue

    cum_comm = calc_cum_comm(data, spec["algo"], common_args["num_clients"], 1.0) * PARAM_TO_GB
    min_len = min(len(cum_comm), len(acc))
    style = line_styles.get(label, {"color": None, "linestyle": "-"})
    total_gb = cum_comm[min_len-1]
    plt.plot(cum_comm[:min_len], acc[:min_len], label=f"{label} ({total_gb:.2f} GB)", linewidth=plot_linewidth, color=style["color"], linestyle=style["linestyle"])

# 绘制 FedAvg 基准线 (使用 epochs=10)
fedavg_data = loader.load("fedavg", **common_args, specific_run=0, keys=["acc"])
if fedavg_data and "acc" in fedavg_data:
    fedavg_acc = max(fedavg_data["acc"])
    fedavg_total_comm = calc_cum_comm(fedavg_data, "fedavg", common_args["num_clients"], 1.0) * PARAM_TO_GB
    style = line_styles.get("FedAvg", {"color": "gray", "linestyle": "--"})
    plt.axhline(y=fedavg_acc, color=style["color"], linestyle=style["linestyle"], linewidth=plot_linewidth, label=f"FedAvg ({fedavg_total_comm[-1]:.2f} GB)")

local_data = loader.load("local", **common_args, specific_run=0, keys=["acc"])
if local_data and "acc" in local_data:
    local_acc = max(local_data["acc"])
    local_total_comm = calc_cum_comm(local_data, "local", common_args["num_clients"], 1.0) * PARAM_TO_GB
    style = line_styles.get("Local", {"color": "black", "linestyle": "--"})
    plt.axhline(y=local_acc, color=style["color"], linestyle=style["linestyle"], linewidth=plot_linewidth, label=f"Local ({local_total_comm[-1]:.2f} GB)")

plt.xlabel("Cumulative Communication (GB)", fontsize=plot_fontsize)
plt.ylabel("Accuracy (%)", fontsize=plot_fontsize)
plt.xlim(0, 100)
plt.ylim(42, 92)
plt.tick_params(labelsize=plot_fontsize)
plt.legend(fontsize=plot_fontsize-4)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("figures/comm_vs_acc.pdf", bbox_inches="tight")
plt.show()

## Fig. 3: scale-free 网络稠密度参数 d 对消融对比

**所需实验：** 包含 $d \in [2, 4, 6, 8, 10, 12, 14, 16, 18]$ 的 DFedSET 实验结果（固定 $\eta=0.9, \lambda_{sa}=20.0, \lambda_{so}=5.0$）。

In [ ]:
# 1. 实验配置与数据加载
d_vals = [2, 4, 6, 8, 10, 12, 14, 16, 18]
sf_ablation_results = {}

for d in d_vals:
    seed_maxes = []
    seed_tr = []
    for s in range(5):
        cfg = {
            **common_args,
            "adj_type": "scale_free",
            "m_scale_free": d,
            "lambda_sa": 20.0,
            "eta": 0.9,
            "lambda_so": 5.0,
            "specific_run": s
        }
        data = loader.load("dfedset", keys=["acc", "num_triggered"], **cfg)
        if data is None:
            continue

        # 记录该 seed 的最大准确率
        acc_data = data["acc"]["model"] if isinstance(data["acc"], dict) else data["acc"]
        seed_maxes.append(max(acc_data))

        # 记录该 seed 的平均触发率
        if "num_triggered" in data:
            tr = np.mean(data["num_triggered"]) / common_args["num_clients"] * 100
        else:
            tr = 100.0
        seed_tr.append(tr)

    if seed_maxes:
        sf_ablation_results[d] = {
            "model_max_mean": np.mean(seed_maxes),
            "model_max_std": np.std(seed_maxes),
            "trig_rate_mean": np.mean(seed_tr),
            "trig_rate_std": np.std(seed_tr)
        }

# 2. 准备绘图数据
sorted_ds = sorted(list(sf_ablation_results.keys()))
acc_mean = [sf_ablation_results[d]["model_max_mean"] for d in sorted_ds]
acc_std = [sf_ablation_results[d]["model_max_std"] for d in sorted_ds]
tr_mean = [sf_ablation_results[d]["trig_rate_mean"] for d in sorted_ds]
tr_std = [sf_ablation_results[d]["trig_rate_std"] for d in sorted_ds]

fig, ax1 = plt.subplots(figsize=(10, 6))

# 主轴 - Accuracy
color = 'tab:blue'
ax1.set_xlabel(r'$d$', fontsize=plot_fontsize)
ax1.set_ylabel('Accuracy (%)', color=color, fontsize=plot_fontsize)
line1 = ax1.plot(sorted_ds, acc_mean, marker='o', color=color, markersize=marker_size, linewidth=plot_linewidth, label='Accuracy')
ax1.fill_between(sorted_ds, np.array(acc_mean) - np.array(acc_std), np.array(acc_mean) + np.array(acc_std), color=color, alpha=0.15)
ax1.tick_params(axis='y', labelcolor=color, labelsize=plot_fontsize)
ax1.tick_params(axis='x', labelsize=plot_fontsize)
ax1.grid(True, alpha=0.3)

# 次轴 - Trigger Rate
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Trigger Rate (%)', color=color, fontsize=plot_fontsize)
line2 = ax2.plot(sorted_ds, tr_mean, marker='s', color=color, markersize=marker_size, linewidth=plot_linewidth, label='Trigger Rate')
ax2.fill_between(sorted_ds, np.array(tr_mean) - np.array(tr_std), np.array(tr_mean) + np.array(tr_std), color=color, alpha=0.15)
ax2.tick_params(axis='y', labelcolor=color, labelsize=plot_fontsize)
ax2.grid(False) # 显式关闭次轴网格

# 固定主次轴 Y 坐标范围显示
ax1.set_ylim(88.2, 89.2)
ax2.set_ylim(65, 95)

# 使用 LinearLocator 限制刻度数量，并确保两边对齐
ax1.yaxis.set_major_locator(mticker.LinearLocator(6))
ax2.yaxis.set_major_locator(mticker.LinearLocator(6))

# 合并图例并展示
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='best', fontsize=plot_fontsize - 2)

fig.tight_layout()
plt.savefig("figures/dfedset_scale_free_ablation.pdf", bbox_inches="tight")
plt.show()

## Fig 4: RSD vs 触发数 vs Round

**所需实验：** 仅需一次完整的 DFedSET run（默认配置，1000 rounds）

In [ ]:
# 前面的数据读取和处理保持原样
data = loader.load("dfedset", **args_dfedset, specific_run=0, keys=["acc", "gsd","num_triggered","triggered_ids"])
if data is None:
    raise RuntimeError("DFedSET 实验结果未找到，请先补跑实验")

x_lim = min(200, len(data["gsd"]))
rsd = np.array(data["gsd"][:x_lim])                     # (R, C)
num_tr = np.array(data["num_triggered"][:x_lim])        # (R,)
acc_model = np.array(data["acc"][:x_lim])      # (R,)

# --- 主图：双子图 (使用 sharex=True 保证对齐) ---
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

color1, color2, color3 = "#4C72B0", "#DD8452", "#55A868"
mean_gsd = rsd.mean(axis=1)
min_gsd = rsd.min(axis=1)
max_gsd = rsd.max(axis=1)

# 计算自适应阈值 (EMA)
eta = 0.9
ema = np.zeros_like(rsd)
ema[0] = 0.5
if x_lim > 1:
    ema[1] = rsd[1]
for r in range(2, x_lim):
    ema[r] = eta * ema[r-1] + (1.0 - eta) * rsd[r]
mean_ema = ema.mean(axis=1)

# --- 上子图：GSD 与自适应阈值 ---
ax1.plot(mean_gsd, color=color1, label="Mean RSD", linewidth=plot_linewidth)
ax1.fill_between(range(x_lim),
    min_gsd, max_gsd,
    alpha=0.15, color=color1)
ax1.plot(mean_ema, color="#E24A33", linestyle="--", label="Mean Threshold", linewidth=plot_linewidth + 0.5)
ax1.set_ylabel("RSD", color="black", fontsize=plot_fontsize)

# 在触发率 50% (0.5) 处添加一条横向参考虚线，但Y轴不显式展示50刻度
ax1.axhline(0.05, color='gray', linewidth=1, alpha=0.1)

# 强制 ax1 的底部严格为 0，使曲线基于坐标轴基线
ax1.set_ylim(0, 0.1)
ax1.set_yticks([0.1])
ax1.grid(True, alpha=0.3)
ax1.tick_params(labelsize=plot_fontsize)
# ax1.axvspan(0, 2, alpha=0.1, color="gray", label="Warmup")
ax1.legend(loc="upper right", fontsize=plot_fontsize-2)

# --- 下子图：触发客户端数 ---
ax2.bar(range(x_lim), num_tr, color=color2, alpha=0.6, width=0.8)
ax2.set_ylabel("# Triggered Clients", color="black", fontsize=plot_fontsize)

# 强制 ax2 的顶部严格为 0
ax2.set_ylim(common_args["num_clients"] + 0.5, 0)
ax2.set_yticks(list(range(0, common_args["num_clients"] + 1, 2)))
ax2.grid(True, alpha=0.3)

# 采用方案4：数值与标签全部放在最底部
ax2.tick_params(labelsize=plot_fontsize)
ax2.set_xlabel("Round", fontsize=plot_fontsize)
ax2.xaxis.set_label_position('bottom')
ax2.set_xlim(0, x_lim)

# 将上下子图严丝合缝拼在一起
fig.subplots_adjust(hspace=0)
fig.savefig("figures/rsd_trigger.pdf", bbox_inches="tight")
plt.show()

## Fig 5: RSD vs 触发数 vs Round

**所需实验：** 仅需一次完整的 DFedSET run（默认配置，1000 rounds）

In [ ]:
# 数据读取：全局阈值消融 (gamma=0.01)
data = loader.load("dfedset", **args_dfedset, specific_run=0, keys=["acc", "gsd","num_triggered","triggered_ids"])
start_round = 5  # 避开前几轮
# 构造触发状态矩阵 (R, C)
trigger_matrix = np.zeros_like(rsd)
for r in range(x_lim):
    for cid in data["triggered_ids"][r]:
        if cid < rsd.shape[1]:
            trigger_matrix[r, cid] = 1.0

fig, (ax_h1, ax_h2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)

# RSD 热力图 (使用 YlOrRd，从下往上为 0--9)
im1 = ax_h1.imshow(rsd[start_round:x_lim].T[::-1, :], aspect="auto", cmap="YlOrRd",
                   extent=[start_round, x_lim, 0, common_args["num_clients"] - 1])
ax_h1.set_yticks(list(range(0, common_args["num_clients"], 5)))
ax_h1.tick_params(labelsize=plot_fontsize + 2)
ax_h1.grid(False)  # 禁用热力图网格线
cbar1 = plt.colorbar(im1, ax=ax_h1, label="RSD", fraction=0.046, pad=0.04)
cbar1.ax.yaxis.label.set_fontsize(plot_fontsize + 2)
cbar1.ax.tick_params(labelsize=plot_fontsize + 2)
ax_h1.text(-0.115, 0.8, "(a)", transform=ax_h1.transAxes, fontsize=plot_fontsize + 2, fontweight="bold", va="bottom", ha="left")

# 触发状态热力图 (使用离散两色映射，从上往下为 0--9)
cmap_discrete = ListedColormap(["#DFF1F1", "#2C5EAD"])
norm_discrete = BoundaryNorm([0, 0.5, 1], cmap_discrete.N)

im2 = ax_h2.imshow(trigger_matrix[start_round:x_lim].T[::-1, :], aspect="auto", cmap=cmap_discrete, norm=norm_discrete,
                   extent=[start_round, x_lim, 0, common_args["num_clients"] - 1])
ax_h2.set_xlabel("Round", fontsize=plot_fontsize + 2)
ax_h2.set_yticks(list(range(0, common_args["num_clients"], 5)))
ax_h2.tick_params(labelsize=plot_fontsize + 2)
ax_h2.grid(False)  # 禁用热力图网格线

cbar2 = plt.colorbar(im2, ax=ax_h2, label="Triggered", fraction=0.046, pad=0.04, ticks=[0.25, 0.75])
cbar2.ax.yaxis.label.set_fontsize(plot_fontsize + 2)
cbar2.ax.tick_params(labelsize=plot_fontsize + 2)
cbar2.ax.set_yticklabels(["0", "1"])

fig.subplots_adjust(hspace=0)
fig.supylabel("Client ID", fontsize=plot_fontsize + 2, x=0.05)
fig.savefig("figures/rsd_heatmap.pdf", bbox_inches="tight")
plt.show()

In [ ]:
data = loader.load("dfedset", ablate_name=None, **args_dfedset, specific_run=0, keys=["triggered_ids"])
if data is not None:
    triggered_ids = data["triggered_ids"]
    counts = [0] * common_args["num_clients"]
    for round_ids in triggered_ids:
        for cid in round_ids:
            if cid < common_args["num_clients"]:
                counts[cid] += 1
    print(f"触发次数 (各客户端): {counts}")

    plt.figure(figsize=(7, 7))
    wedges, texts = plt.pie(
        counts,
        labels=[str(i) for i in range(common_args["num_clients"])],
        colors=plt.cm.tab20(np.linspace(0, 1, common_args["num_clients"])),
        labeldistance=0.75,
        textprops={'fontsize': 2 * plot_fontsize, 'fontweight': 'bold', 'color': 'black'},
    )
    # plt.text(-0.73, 0.9, f"Adaptive", ha='center', va='center', fontsize=plot_fontsize-2, fontweight='bold')
    plt.axis('off')
    plt.xlim(-1.0, 1.0)
    plt.ylim(-1.0, 1.0)
    for t in texts:
        t.set_horizontalalignment('center')
        t.set_verticalalignment('center')
    plt.tight_layout()
    plt.savefig("figures/trigger_pie_base.pdf", bbox_inches="tight", pad_inches=0.01)
    plt.show()

In [ ]:
# 数据读取：全局阈值消融 (gamma=0.01)
data = loader.load("dfedset", ablate_name="trigger_global_gamma_0.001",
                   **args_dfedset, specific_run=0, keys=["acc", "gsd", "num_triggered", "triggered_ids"])
if data is None:
    raise RuntimeError("DFedSET 实验结果未找到，请先补跑实验")

x_lim = min(200, len(data["gsd"]))
rsd = np.array(data["gsd"][:x_lim])                     # (R, C)
num_tr = np.array(data["num_triggered"][:x_lim])        # (R,)
acc_model = np.array(data["acc"][:x_lim])      # (R,)

start_round = 5  # 避开前几轮
# 构造触发状态矩阵 (R, C)
trigger_matrix = np.zeros_like(rsd)
for r in range(x_lim):
    for cid in data["triggered_ids"][r]:
        if cid < rsd.shape[1]:
            trigger_matrix[r, cid] = 1.0

fig, (ax_h1, ax_h2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)

# RSD 热力图 (使用 YlOrRd，从下往上为 0--9)
im1 = ax_h1.imshow(rsd[start_round:x_lim].T[::-1, :], aspect="auto", cmap="YlOrRd",
                   extent=[start_round, x_lim, 0, common_args["num_clients"] - 1])
ax_h1.set_yticks(list(range(0, common_args["num_clients"], 5)))
ax_h1.tick_params(labelsize=plot_fontsize + 2)
ax_h1.grid(False)  # 禁用热力图网格线
cbar1 = plt.colorbar(im1, ax=ax_h1, label="RSD", fraction=0.046, pad=0.04)
cbar1.ax.yaxis.label.set_fontsize(plot_fontsize + 2)
cbar1.ax.tick_params(labelsize=plot_fontsize + 2)
ax_h1.text(-0.115, 0.8, "(b)", transform=ax_h1.transAxes, fontsize=plot_fontsize + 2, fontweight="bold", va="bottom", ha="left")

# 触发状态热力图 (使用离散两色映射，从上往下为 0--9)
cmap_discrete = ListedColormap(["#DFF1F1", "#2C5EAD"])
norm_discrete = BoundaryNorm([0, 0.5, 1], cmap_discrete.N)

im2 = ax_h2.imshow(trigger_matrix[start_round:x_lim].T[::-1, :], aspect="auto", cmap=cmap_discrete, norm=norm_discrete,
                   extent=[start_round, x_lim, 0, common_args["num_clients"] - 1])
ax_h2.set_xlabel("Round", fontsize=plot_fontsize + 2)
ax_h2.set_yticks(list(range(0, common_args["num_clients"], 5)))
ax_h2.tick_params(labelsize=plot_fontsize + 2)
ax_h2.grid(False)  # 禁用热力图网格线

cbar2 = plt.colorbar(im2, ax=ax_h2, label="Triggered", fraction=0.046, pad=0.04, ticks=[0.25, 0.75])
cbar2.ax.yaxis.label.set_fontsize(plot_fontsize + 2)
cbar2.ax.tick_params(labelsize=plot_fontsize + 2)
cbar2.ax.set_yticklabels(["0", "1"])

fig.subplots_adjust(hspace=0)
fig.supylabel("Client ID", fontsize=plot_fontsize + 2, x=0.05)
fig.savefig("figures/rsd_heatmap_global_0.01.pdf", bbox_inches="tight")
plt.show()

In [ ]:
data = loader.load("dfedset", ablate_name="trigger_global_gamma_0.001", **args_dfedset, specific_run=0, keys=["triggered_ids"])
if data is not None:
    triggered_ids = data["triggered_ids"]
    counts = [0] * common_args["num_clients"]
    for round_ids in triggered_ids:
        for cid in round_ids:
            if cid < common_args["num_clients"]:
                counts[cid] += 1
    print(f"触发次数 (各客户端): {counts}")

    plt.figure(figsize=(7, 7))
    wedges, texts = plt.pie(
        counts,
        labels=[str(i) for i in range(common_args["num_clients"])],
        colors=plt.cm.tab20(np.linspace(0, 1, common_args["num_clients"])),
        labeldistance=0.75,
        textprops={'fontsize': 2 * plot_fontsize, 'fontweight': 'bold', 'color': 'black'},
    )
    # plt.text(-0.8, 0.9, "G(0.01)", ha='center', va='center', fontsize=plot_fontsize-2, fontweight='bold')
    plt.axis('off')
    plt.xlim(-1.0, 1.0)
    plt.ylim(-1.0, 1.0)
    for t in texts:
        t.set_horizontalalignment('center')
        t.set_verticalalignment('center')
    plt.tight_layout()
    plt.savefig("figures/trigger_pie_g001.pdf", bbox_inches="tight", pad_inches=0.01)
    plt.show()

In [ ]:
# 数据读取：全局阈值消融 (gamma=0.05)
data = loader.load("dfedset", ablate_name="trigger_global_gamma_0.005",
                   **args_dfedset, specific_run=0, keys=["acc", "gsd", "num_triggered", "triggered_ids"])
if data is None:
    raise RuntimeError("DFedSET 实验结果未找到，请先补跑实验")

x_lim = min(200, len(data["gsd"]))
rsd = np.array(data["gsd"][:x_lim])                     # (R, C)
num_tr = np.array(data["num_triggered"][:x_lim])        # (R,)
acc_model = np.array(data["acc"][:x_lim])      # (R,)

start_round = 5  # 避开前几轮
# 构造触发状态矩阵 (R, C)
trigger_matrix = np.zeros_like(rsd)
for r in range(x_lim):
    for cid in data["triggered_ids"][r]:
        if cid < rsd.shape[1]:
            trigger_matrix[r, cid] = 1.0

fig, (ax_h1, ax_h2) = plt.subplots(2, 1, figsize=(10, 4), sharex=True)

# RSD 热力图 (使用 YlOrRd，从下往上为 0--9)
im1 = ax_h1.imshow(rsd[start_round:x_lim].T[::-1, :], aspect="auto", cmap="YlOrRd",
                   extent=[start_round, x_lim, 0, common_args["num_clients"] - 1])
ax_h1.set_yticks(list(range(0, common_args["num_clients"], 5)))
ax_h1.tick_params(labelsize=plot_fontsize + 2)
ax_h1.grid(False)  # 禁用热力图网格线
cbar1 = plt.colorbar(im1, ax=ax_h1, label="RSD", fraction=0.046, pad=0.04)
cbar1.ax.yaxis.label.set_fontsize(plot_fontsize + 2)
cbar1.ax.tick_params(labelsize=plot_fontsize + 2)
ax_h1.text(-0.115, 0.8, "(c)", transform=ax_h1.transAxes, fontsize=plot_fontsize + 2, fontweight="bold", va="bottom", ha="left")

# 触发状态热力图 (使用离散两色映射，从上往下为 0--9)
cmap_discrete = ListedColormap(["#DFF1F1", "#2C5EAD"])
norm_discrete = BoundaryNorm([0, 0.5, 1], cmap_discrete.N)

im2 = ax_h2.imshow(trigger_matrix[start_round:x_lim].T[::-1, :], aspect="auto", cmap=cmap_discrete, norm=norm_discrete,
                   extent=[start_round, x_lim, 0, common_args["num_clients"] - 1])
ax_h2.set_xlabel("Round", fontsize=plot_fontsize + 2)
ax_h2.set_yticks(list(range(0, common_args["num_clients"], 5)))
ax_h2.tick_params(labelsize=plot_fontsize + 2)
ax_h2.grid(False)  # 禁用热力图网格线

cbar2 = plt.colorbar(im2, ax=ax_h2, label="Triggered", fraction=0.046, pad=0.04, ticks=[0.25, 0.75])
cbar2.ax.yaxis.label.set_fontsize(plot_fontsize + 2)
cbar2.ax.tick_params(labelsize=plot_fontsize + 2)
cbar2.ax.set_yticklabels(["0", "1"])

fig.subplots_adjust(hspace=0)
fig.supylabel("Client ID", fontsize=plot_fontsize + 2, x=0.05)
fig.savefig("figures/rsd_heatmap_global_0.05.pdf", bbox_inches="tight")
plt.show()

In [ ]:
data = loader.load("dfedset", ablate_name="trigger_global_gamma_0.005", **args_dfedset, specific_run=0, keys=["triggered_ids"])
if data is not None:
    triggered_ids = data["triggered_ids"]
    counts = [0] * common_args["num_clients"]
    for round_ids in triggered_ids:
        for cid in round_ids:
            if cid < common_args["num_clients"]:
                counts[cid] += 1
    print(f"触发次数 (各客户端): {counts}")

    plt.figure(figsize=(7, 7))
    wedges, texts = plt.pie(
        counts,
        labels=[str(i) for i in range(common_args["num_clients"])],
        colors=plt.cm.tab20(np.linspace(0, 1, common_args["num_clients"])),
        labeldistance=0.75,
        textprops={'fontsize': 2 * plot_fontsize, 'fontweight': 'bold', 'color': 'black'},
    )
    # plt.text(-0.8, 0.9, "G(0.05)", ha='center', va='center', fontsize=plot_fontsize-2, fontweight='bold')
    plt.axis('off')
    plt.xlim(-1.0, 1.0)
    plt.ylim(-1.0, 1.0)
    for t in texts:
        t.set_horizontalalignment('center')
        t.set_verticalalignment('center')
    plt.tight_layout()
    plt.savefig("figures/trigger_pie_g005.pdf", bbox_inches="tight", pad_inches=0.01)
    plt.show()

## Fig 6: 超参数分析 — λ_sa × λ_so 热力图

**所需实验：** 单 λ_sa（固定 20），λ_so 线性扫描
遍历 λ_sa ∈ {20}, λ_so ∈ {0, 1, 2, ..., 10}
DFedSET, cifar10, epochs=1, alpha=0.1

In [ ]:
lambda_sa_list = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
lambda_so_list = [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20]
model_acc_grid = np.full((len(lambda_sa_list), len(lambda_so_list)), np.nan)

for i, l_sa in enumerate(lambda_sa_list):
    for j, l_so in enumerate(lambda_so_list):
        args = {**common_args, "lambda_sa": l_sa, "eta": 0.9, "lambda_so": l_so}
        seed_maxes = []
        for s in range(5):
            data = loader.load("dfedset", **args, specific_run=s, keys=["acc"])
            if data is not None and "acc" in data:
                seed_maxes.append(max(data["acc"]))
        if seed_maxes:
            model_acc_grid[i, j] = np.mean(seed_maxes)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(model_acc_grid, cmap="viridis", aspect="auto")
ax.grid(False)
ax.set_xticks(range(len(lambda_so_list)))
ax.set_xticklabels(lambda_so_list, fontsize=plot_fontsize)
ax.set_yticks(range(len(lambda_sa_list)))
ax.set_yticklabels(lambda_sa_list, fontsize=plot_fontsize)
ax.set_xlabel(r"$\lambda_{so}$", fontsize=plot_fontsize)
ax.set_ylabel(r"$\lambda_{sa}$", fontsize=plot_fontsize)
cbar = plt.colorbar(im, ax=ax, fraction=0.046)
cbar.ax.tick_params(labelsize=plot_fontsize)
for ii in range(len(lambda_sa_list)):
    for jj in range(len(lambda_so_list)):
        val = model_acc_grid[ii, jj]
        if not np.isnan(val):
            color = "w" if val < np.nanmax(model_acc_grid) * 0.7 else "k"
            diff = val - model_acc_grid[0, 0]
            ax.text(jj, ii, f"{diff:+.2f}", ha="center", va="center", color=color, fontsize=plot_fontsize-4)

plt.tight_layout()
plt.savefig("figures/hyperparam_heatmap.pdf", bbox_inches="tight")
plt.show()

## Fig. 7: 不同 eta 值的消融实验参数对比

**所需实验：** 包含 $\eta = 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99, 0.999$ 的 DFedSET 实验结果（固定 $\lambda_{sa}=20.0, \lambda_{so}=5.0$）。

In [ ]:
# 1. 实验配置与数据加载
eta_vals = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99, 0.999]
eta_ablation_results = {}

for eta in eta_vals:
    seed_maxes = []
    seed_tr = []
    for s in range(5):
        cfg = {
            **common_args,
            "lambda_sa": 20.0,
            "eta": eta,
            "lambda_so": 5.0,
            "specific_run": s
        }
        data = loader.load("dfedset", keys=["acc", "num_triggered"], **cfg)
        if data is None:
            continue

        # 记录该 seed 的最大准确率
        acc_data = data["acc"]["model"] if isinstance(data["acc"], dict) else data["acc"]
        seed_maxes.append(max(acc_data))

        # 记录该 seed 的平均触发率
        if "num_triggered" in data:
            tr = np.mean(data["num_triggered"]) / common_args["num_clients"] * 100
        else:
            tr = 100.0
        seed_tr.append(tr)

    if seed_maxes:
        eta_ablation_results[eta] = {
            "model_max_mean": np.mean(seed_maxes),
            "model_max_std": np.std(seed_maxes),
            "trig_rate_mean": np.mean(seed_tr),
            "trig_rate_std": np.std(seed_tr)
        }

# 2. 准备绘图数据
sorted_etas = sorted(list(eta_ablation_results.keys()))
acc_mean = [eta_ablation_results[eta]["model_max_mean"] for eta in sorted_etas]
acc_std = [eta_ablation_results[eta]["model_max_std"] for eta in sorted_etas]
tr_mean = [eta_ablation_results[eta]["trig_rate_mean"] for eta in sorted_etas]
tr_std = [eta_ablation_results[eta]["trig_rate_std"] for eta in sorted_etas]

fig, ax1 = plt.subplots(figsize=(10, 6))

# 主轴 - Accuracy
color = 'tab:blue'
ax1.set_xlabel(r'$\eta$', fontsize=plot_fontsize)
ax1.set_ylabel('Accuracy (%)', color=color, fontsize=plot_fontsize)
line1 = ax1.plot(sorted_etas, acc_mean, marker='o', color=color, markersize=marker_size, linewidth=plot_linewidth, label='Accuracy')
ax1.fill_between(sorted_etas, np.array(acc_mean) - np.array(acc_std), np.array(acc_mean) + np.array(acc_std), color=color, alpha=0.15)
ax1.tick_params(axis='y', labelcolor=color, labelsize=plot_fontsize)
ax1.tick_params(axis='x', labelsize=plot_fontsize)
ax1.grid(True, alpha=0.3)

# 次轴 - Trigger Rate
ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Trigger Rate (%)', color=color, fontsize=plot_fontsize)
line2 = ax2.plot(sorted_etas, tr_mean, marker='s', color=color, markersize=marker_size, linewidth=plot_linewidth, label='Trigger Rate')
ax2.fill_between(sorted_etas, np.array(tr_mean) - np.array(tr_std), np.array(tr_mean) + np.array(tr_std), color=color, alpha=0.15)
ax2.tick_params(axis='y', labelcolor=color, labelsize=plot_fontsize)
ax2.grid(False) # 显式关闭次轴网格

# 固定主次轴 Y 坐标范围显示
ax1.set_ylim(84, 88.5)
ax2.set_ylim(0, 90)

# 使用 LinearLocator 限制刻度数量，并确保两边对齐
ax1.yaxis.set_major_locator(mticker.LinearLocator(6))
ax2.yaxis.set_major_locator(mticker.LinearLocator(6))

# 合并图例并展示
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax1.legend(lines, labels, loc='best', fontsize=plot_fontsize - 2)

fig.tight_layout()
plt.savefig("figures/dfedset_eta_ablation_double_y.pdf", bbox_inches="tight")
plt.show()